In [4]:
import os
import requests
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import mlflow

RETRAIN = False
DATA_PATH = "../../02_data/mnist"
NUM_CLIENTS = 3

mlflow.set_tracking_uri("http://localhost:5100")
MLFLOW_EXPERIMENT_NAME = "edge-fl-experiment"

In [5]:
#!docker pull ghcr.io/bencefarkas27/aggregator:test-auth
!docker run --name aggregator_test -d --rm -p 5100:5000 -p 9100:8000  ghcr.io/bencefarkas27/aggregator:test-auth

ac3c7045c7461378aba883e5d5d1f32c1a282f75397d456ddad0b632786d36ac


### Healt endpoint check

In [9]:
response = requests.get("http://localhost:9100/health")
print(f"Health check response of aggregator: {response.status_code} - {response.text}")

Health check response of aggregator: 200 - 


In [ ]:
x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
print(f"x_train shape: {x_train.shape[1:]}")

In [11]:
run_ids = []

def create_dummy_model(input_shape):
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=input_shape),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_client(client_id):
    # Load client data
    x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
    y_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_labels.npy")
    # Create and train the model
    model = create_dummy_model(input_shape=x_train.shape[1:])
    model.fit(x_train, y_train, epochs=3)
    # Save the trained model weights
    model.save_weights(f"dummy_models/client_{client_id}.weights.h5")

def mlflow_log_client_model(client_id):
    weights_path = f"dummy_models/client_{client_id}.weights.h5"
    x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
    input_shape = x_train.shape[1:]
    model = create_dummy_model(input_shape=input_shape)
    model.load_weights(weights_path)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    with mlflow.start_run(run_name=f"client_{client_id}_run") as run:
        # Log the model to MLflow
        mlflow.tensorflow.log_model(model, f"client_{client_id}_model")
        run_ids.append(run.info.run_id)


In [12]:
# Run these for client model training and logging to MLflow if no volume mounted client models exist for mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "admin"
with open("../../04_deploy/aggregator/cfg/mlflow_admin.json", "r") as f:
                admin_config = json.load(f)
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["admin_password"]
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
for client_id in range(NUM_CLIENTS):
    if RETRAIN:
        train_client(client_id)
    mlflow_log_client_model(client_id)

/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/07/24 10:48:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/24 10:48:57 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_0_run at: http://localhost:5100/#/experiments/1/runs/3d3530d412c04c24843267896bcf3801
🧪 View experiment at: http://localhost:5100/#/experiments/1


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_2', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/07/24 10:49:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/24 10:49:15 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_1_run at: http://localhost:5100/#/experiments/1/runs/ac22949858584b0aa18803bde6e81f21
🧪 View experiment at: http://localhost:5100/#/experiments/1


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_4', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/07/24 10:49:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/24 10:49:29 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_2_run at: http://localhost:5100/#/experiments/1/runs/1cda1343736745db98a247fe2d8886c6
🧪 View experiment at: http://localhost:5100/#/experiments/1


In [13]:
# Get run ids from experiment
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
runs_df = mlflow.search_runs([experiment.experiment_id])
#run_ids = runs_df["run_id"].tolist()

runs = [f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)]
print(runs)
print(run_ids)

['runs:/3d3530d412c04c24843267896bcf3801/client_0_model', 'runs:/ac22949858584b0aa18803bde6e81f21/client_1_model', 'runs:/1cda1343736745db98a247fe2d8886c6/client_2_model']
['3d3530d412c04c24843267896bcf3801', 'ac22949858584b0aa18803bde6e81f21', '1cda1343736745db98a247fe2d8886c6']


### Calling do_aggregate widt fedavg

In [14]:
aggregation_config = {
    "local_model_hosts": [
        "http://localhost:5000"
        for _ in range(NUM_CLIENTS)
    ],
    "local_model_uris": [
        f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)
    ],
    "local_data_size": [
        len(np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy"))
        for client_id in range(NUM_CLIENTS)
    ],
    "aggregation_algorithm": "fedavg",
    
}

response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)

In [15]:
print(response.json())

{'aggregated_model_host_external': 'http://localhost:5100', 'aggregated_model_host_organizational': 'http://aggregator:5000', 'aggregated_model_uri': 'models:/m-8e40a939b0934e8aaec928c56ce75250', 'aggregated_model_username': 'model_user', 'aggregated_model_password': 'hiPMMENYuzajkO_vuq9owKnV3ORxRSapm7GgpNw9BAU'}


In [16]:
# Testing the client models vs the aggregated model
# Load test data
x_test = np.load(f"{DATA_PATH}/test_images.npy")
y_test = np.load(f"{DATA_PATH}/test_labels.npy")

In [17]:
# Pull the aggregated model from mlflow
try:
    # Search the experiment and retrieve the runs
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    models = mlflow.search_logged_models(
        experiment_ids=[experiment.experiment_id],
        order_by=[{"field_name": "creation_time", "ascending": False}],
        output_format="list"
    )
    latest_model = models[0]
    aggregated_model = mlflow.tensorflow.load_model(f"models:/{latest_model.model_id}")
    aggregated_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

    print(f"Loaded latest aggregated model: {latest_model.model_id}")

    aggregated_loss, aggregated_accuracy = aggregated_model.evaluate(x_test, y_test, verbose=0)
    print(f"Aggregated Model - Loss: {aggregated_loss}, Accuracy: {aggregated_accuracy}")
except Exception as e:
    print(f"Failed to load model: {e}")


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Loaded latest aggregated model: m-8e40a939b0934e8aaec928c56ce75250
Aggregated Model - Loss: 0.6987630724906921, Accuracy: 0.894599974155426


In [18]:
client_losses = []
client_accuracies = []
for client_id in range(NUM_CLIENTS):
    # Load the client model
    client_model = create_dummy_model(input_shape=x_test.shape[1:])
    client_model.load_weights(f"dummy_models/client_{client_id}.weights.h5")
    
    # Evaluate the client model
    client_loss, client_accuracy = client_model.evaluate(x_test, y_test, verbose=0)
    print(f"Client {client_id} Model - Loss: {client_loss}, Accuracy: {client_accuracy}")
    client_losses.append(client_loss)
    client_accuracies.append(client_accuracy)

/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_7', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 0 Model - Loss: 0.7646135687828064, Accuracy: 0.8902999758720398


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_8', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 1 Model - Loss: 0.7928179502487183, Accuracy: 0.8978999853134155


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_9', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 2 Model - Loss: 0.6382331252098083, Accuracy: 0.8949000239372253


In [19]:
print(f"Aggregator's loss is {aggregated_loss} vs clients's losses: {client_losses} and avarage client loss: {np.mean(client_losses)}")
print(f"Aggregator's accuracy is {aggregated_accuracy} vs clients's accuracies: {client_accuracies} and avarage client accuracy: {np.mean(client_accuracies)}")

Aggregator's loss is 0.6987630724906921 vs clients's losses: [0.7646135687828064, 0.7928179502487183, 0.6382331252098083] and avarage client loss: 0.731888214747111
Aggregator's accuracy is 0.894599974155426 vs clients's accuracies: [0.8902999758720398, 0.8978999853134155, 0.8949000239372253] and avarage client accuracy: 0.8943666617075602


### Calling do_aggregate widt deltarep

In [ ]:
ggregation_config = {
    "local_model_hosts": [
        "http://localhost:5000"
        for _ in range(NUM_CLIENTS)
    ],
    "local_model_uris": [
        f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)
    ],
    
    "aggregation_algorithm": "deltarep",
    
}

response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)

In [12]:
!docker stop aggregator_test

aggregator_test
